# Plan de colonnes et axes d'analyse

---

## 1. Objectif du notebook

---

Ce notebook a pour objectif de définir le plan analytique du projet :

- Structuration des variables par axe thématique,
- Justification des choix de variables retenues ou exclues,
- Cadrage méthodologique de l'analyse exploratoire.

La préparation spécifique des données est réalisée directement dans chaque notebook d'analyse afin de préserver la lisibilité par axe.



## 2. Définition des axes d'analyse

---

Dans cette section, j'analyse les colonnes du dataset afin de les regrouper en trois axes thématiques :

- Conditions personnelles
- Conditions de travail
- Satisfaction et implication.

Ce travail permet de structurer les futures analyses.


**Axe : Conditions personnelles (Facteurs externes individuels)** 

Cet axe regroupe les caractéristiques individuelles qui ne dépendent pas de l'entreprise.
- Style de vie
- Stabilité personnelle
- Parcours éducatif
- Situation familiale

Ces variables sont considérées comme des facteurs externes individuels pouvant influencer l'attrition.


**Axe : Conditions de travail (Facteurs externes organisationnels)**

Cet axe regroupe les éléments organisationnels contrôlés par l'entreprise :
- Salaire
- Charge de travail
- Rôle occupé
- Horaire
- Progression interne 
- Relation avec le manager
- Formation.

Ces facteurs peuvent être influencés par des politiques RH.


**Axe : Satisfaction et implication (Facteurs internes psychologiques)**

Cet axe représente le ressenti interne du salarié :
- Satisfaction au travail
- Satisfaction de l'environnement
- Satisfaction des relations professionnelles
- Équilibre vie pro / perso
- Implication et engagement au travail.

Cet axe sera analysé comme un facteur explicatif potentiel de l'attrition, puis analysé en lien avec les facteurs externes afin d'identifier les leviers potentiels.



## 3. Plan de colonnes

---

Ce notebook génère un fichier “plan de colonnes” listant l'ensemble des variables du dataset.

Pour chaque variable, deux informations sont renseignées :
- **axe** : rattachement thématique (Conditions personnelles / Conditions de travail / Satisfaction et implication / Technique / Attrition).

- **garder** : conservation de la variable pour l'analyse (oui / non)


Le workflow est le suivant :
1. Création automatique d'un fichier `plan_colonnes.csv` (squelette) si celui-ci n'existe pas.
2. Remplissage manuel du plan (axe + garder).
3. Sauvegarde finale sous `plan_colonnes_done.csv`, utilisée ensuite comme référence dans les notebooks d'analyse.


In [ ]:
# Import librairies
import pandas as pd
import os

from pathlib import Path
from IPython.display import display_html

In [ ]:
# Définition des chemins

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
INTERIM_DATA_DIR = DATA_DIR / "interim"
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True) # Créer le dossier interim

PROCESSED_FILE = PROCESSED_DATA_DIR / "employees_clean.parquet"

PLAN_FILE = INTERIM_DATA_DIR / "plan_colonnes.csv"          # Fichier à remplir dans Excel
DONE_FILE = INTERIM_DATA_DIR / "plan_colonnes_done.csv"     # Fichier final 

In [3]:

# Chargement du dataset
df = pd.read_parquet(PROCESSED_FILE)

# Si le DONE existe : on le charge
if DONE_FILE.exists():
    df_col = pd.read_csv(DONE_FILE, sep=",")
    print("Chargé : plan_colonnes_done.csv ")

# Sinon : on s'assure que le PLAN existe, sinon on le crée une seule fois
else:
    if not PLAN_FILE.exists():
        df_col = pd.DataFrame({"col_initial": df.columns, "axe": "", "garder": ""}) # Créer tableau
       
        df_col.to_csv(PLAN_FILE, index=False)
        print(" Créé : plan_colonnes.csv (à compléter dans Excel)")
    else:
        print(" plan_colonnes.csv existe déjà")

    df_col = pd.read_csv(PLAN_FILE)

df_col.head()



Chargé : plan_colonnes_done.csv 


,col_initial,axe,garder
0,Age,Conditions personnelles,oui
1,Attrition,Attrition,oui
2,BusinessTravel,Conditions de travail,oui
3,DailyRate,Conditions de travail,non
4,Department,Conditions de travail,oui


**Sélection des variables pour l'analyse**

Les vérifications suivantes permettent de :
- vérifier le nombre de variables conservées ou écartées après le processus de sélection,
- analyser la répartition des variables conservées par axe d'analyse.

Cela garantit la cohérence et la traçabilité des choix effectués lors de la préparation des données.


In [ ]:
var_conserver = df_col["garder"].value_counts().to_frame(name="Nombre")
repartition_par_axe = df_col[df_col["garder"]=="oui"]["axe"].value_counts().to_frame(name="Nombre")

display_html(
    """
    <div style="display: inline-flex; gap: 60px; align-items: flex-start;">
        <div>
            <h4> Nombre de variables conservées / écartées</h4>
            {t1}
        </div>
        <div>
            <h4> Répartition des variables conservées par axe</h4>
            {t2}
        </div>
    </div>
    """.format(
        t1=var_conserver.to_html(),
        t2=repartition_par_axe.to_html()),raw=True)


,Nombre
garder,
oui,29
non,6
,Nombre
axe,
Conditions de travail,14
Vie personnelle,7
Satisfaction et Implication,6
Attrition,1
Technique,1


La majorité des variables conservées concerne les conditions de travail, reflétant la richesse de cet axe dans le dataset et son importance dans l'analyse de l'attrition. Les axes conditions personnelles et satisfaction/implication restent suffisamment représentés pour permettre une lecture équilibrée et transversale des facteurs associés au départ.

---

## 4. Justification des exclusions

---

Certaines variables ont été exclues afin de limiter la redondance et le bruit analytique :

- **Variables de rémunération redondantes** (DailyRate, HourlyRate, MonthlyRate)  
MonthlyIncome est privilégiée, car plus interprétable et plus standard en contexte RH.

- **Variables techniques ou constantes** (EmployeeCount, Over18, StandardHours)  
Elles n'apportent aucune information discriminante et sont donc hors champ d'analyse.

**Remarque : EmployeeNumber**  

EmployeeNumber est conservée uniquement comme identifiant technique afin de permettre la fusion des DataFrames d'axes lors des étapes ultérieures. Elle n'est pas utilisée comme variable explicative.



In [5]:
df_col.loc[df_col["garder"] == "non"]

,col_initial,axe,garder
3,DailyRate,Conditions de travail,non
8,EmployeeCount,Technique,non
12,HourlyRate,Conditions de travail,non
19,MonthlyRate,Conditions de travail,non
21,Over18,Technique,non
26,StandardHours,Technique,non



## 5. Conclusion

---

À ce stade, le plan de colonnes est finalisé et servira de référence pour la sélection des variables dans les notebooks d'analyse par axe. Les exclusions visent principalement à réduire la redondance (variables de rémunération exprimées à différentes granularités) et à retirer les variables techniques/constantes. EmployeeNumber est conservée uniquement comme identifiant technique pour les fusions transversales.